# DataForge Arena - GRPO Training on Colab

**Meta PyTorch + HuggingFace OpenEnv Hackathon 2026**

This notebook trains an LLM to repair corrupted enterprise data using Group Relative Policy Optimization (GRPO).

Requirements: GPU runtime (T4 minimum). Go to **Runtime -> Change runtime type -> T4 GPU**.

## 1. Setup & Install

In [ ]:
# Clone repo (already has all bug fixes)
!git clone https://github.com/vivekyarra/dataforge-arena.git /content/dataforge-arena
%cd /content/dataforge-arena

In [ ]:
# Install dependencies
!pip install -q unsloth trl transformers datasets accelerate bitsandbytes openenv faker mergekit llm-blender
print('Dependencies installed')

In [ ]:
# Generate clean training data
!python training/generate_data.py

## 2. Verify GPU & Model Selection

In [ ]:
!nvidia-smi
print('---')
!python -c "from training.model_config import detect_gpu, select_model; g=detect_gpu(); m=select_model(g); print(f'GPU: {g}'); print(f'Model: {m[\"label\"]}')"

## 3. Run Tests (must all pass)

In [ ]:
!python -m pytest -q

## 4. Quick Sanity Check - Single Episode

In [ ]:
import pandas as pd
from environment.env import DataForgeEnv, SurgeonAction
from environment.corruptor import Corruptor
from environment.schemas import HEALTHCARE_SCHEMA

df = pd.read_csv('data/healthcare_clean.csv')
env = DataForgeEnv(Corruptor(), HEALTHCARE_SCHEMA, df)
obs = env.reset()
action = SurgeonAction(reasoning='test null', tool_id=0, column=2, row_id=0)
obs2, reward, done, info = env.step(action)
print(f'Reward: {reward}')
print(f'Done: {done}')
print('Episode loop works!')

## 5. GRPO Training

This runs the full training loop. On a T4: ~60 min for 80 steps.

In [ ]:
# Suppress noisy warnings
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)

# Run training
!python training/train_grpo.py

## 6. Evaluation - Before/After Numbers

In [ ]:
!python eval/evaluate.py --episodes 20 --tier 1
print('---')
!python eval/evaluate.py --episodes 10 --tier 2
print('---')
!python eval/evaluate.py --episodes 10 --tier 3

## 7. Training Curves

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

try:
    log = pd.read_csv('logs/training_log.csv')
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.patch.set_facecolor('#0a0a0f')
    
    for ax in axes:
        ax.set_facecolor('#111118')
        ax.tick_params(colors='#94a3b8')
        ax.spines['bottom'].set_color('#1e293b')
        ax.spines['left'].set_color('#1e293b')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
    
    axes[0].plot(log['step'], log['total_reward'], color='#10b981', linewidth=2)
    axes[0].set_title('Total Reward', color='#e2e8f0', fontsize=14)
    axes[0].set_xlabel('Step', color='#94a3b8')
    axes[0].set_ylabel('Reward', color='#94a3b8')
    axes[0].axhline(y=0, color='#374151', linestyle='--', alpha=0.5)
    
    axes[1].plot(log['step'], log['difficulty'], color='#f59e0b', linewidth=2)
    axes[1].set_title('Difficulty Escalation', color='#e2e8f0', fontsize=14)
    axes[1].set_xlabel('Step', color='#94a3b8')
    axes[1].set_ylabel('Tier', color='#94a3b8')
    
    plt.tight_layout()
    plt.savefig('training_curves.png', dpi=150, bbox_inches='tight', facecolor='#0a0a0f')
    plt.show()
    print('Saved to training_curves.png')
except FileNotFoundError:
    print('No training log found. Run training first.')

## 8. Save & Download

In [ ]:
# Zip the trained model for download
!zip -r /content/dataforge-trained.zip outputs/dataforge-surgeon/ logs/ eval/results.json training_curves.png 2>/dev/null || true
print('Download: /content/dataforge-trained.zip')

# Show final stats
import json
try:
    with open('eval/results.json') as f:
        r = json.load(f)
    print(f"\nSurgeon advantage accuracy delta: {r['surgeon_advantage_accuracy_delta']*100:+.2f}% over random baseline")
except: pass